# Manual sampling exploration

Current code includes the corrections documented in [research notes](../docs/research-notes.md). Historical evidence remains in [the report](../docs/report.pdf) and exported figures; reruns can differ.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src/samplers.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from samplers import uniform, exponential, negative_binomial, generate_normal_box_muller, gamma_rejection

import numpy as np
np.random.seed(42)


In [ ]:
import math
import numpy as np
import pandas as pd
from math import gamma, sqrt, pi
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import chi2

Bernoulli from uniform


In [ ]:
def bernoulli(p,num_samples,trial_runs):
    bernoulli_sample = (np.random.rand(num_samples, trial_runs) < p).astype(int)
    return bernoulli_sample

p = 0.3         
num_samples = 10000       


X = bernoulli(p, num_samples,trial_runs=1)      
flat_X = X.flatten()                                
counts = np.bincount(flat_X, minlength=2) / flat_X.size

x_vals = np.array([0, 1])
pmf = np.array([1 - p, p])                  


plt.figure(figsize=(6,4))
plt.bar(x_vals - 0.15, pmf, width=0.3, label='Theoretical PDF')
plt.bar(x_vals + 0.15, counts, width=0.3, label='Simulation Proportions')
plt.xlabel('Value')
plt.ylabel('Probability')
plt.xticks(x_vals)
plt.legend()
plt.title(f'Bernoulli(p={p}): Theoretical vs. Empirical (n={num_samples})')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

Binomial distribution (from Bernoulli)


In [ ]:
N, n, p = 10000, 10, 0.3
samples = (np.random.rand(N, n) < p).sum(axis=1)

counts = np.bincount(samples, minlength=n+1) / N
k = np.arange(n+1)
true_pmf = [math.comb(n, ki) * p**ki * (1-p)**(n-ki) for ki in k]

plt.bar(k, counts, alpha=0.7, label='Sample frequencies')
plt.plot(k, true_pmf, 'o-', linewidth=2, label='True PMF')
plt.title(f'Binomial(n={n}, p={p}) via Bernoulli Sum')
plt.xlabel('k')
plt.ylabel('Probability')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Geometric distribution


In [ ]:
import numpy as np

def sample_geometric(n, p):
    """
    Draw n samples from a Geometric(p) distribution
    using inverse transform sampling.
    Returns integers in {1,2,3,...}.
    """
    # Uniform draws
    U = np.random.rand(n)
    # Inverse CDF transform
    X = np.floor(np.log1p(-U) / np.log1p(-p)).astype(int) + 1
    return X

# Example usage:
# samples = sample_geometric(10000, 0.3)
import numpy as np
import matplotlib.pyplot as plt

# Parameters
p = 0.3
n = 10000

# Sample
samples = sample_geometric(n, p)

# Theoretical PMF
k = np.arange(1, samples.max() + 1)
pmf = (1 - p)**(k - 1) * p

# Empirical frequencies (as probabilities)
counts, _ = np.histogram(samples, bins=np.arange(1, k.max() + 2) - 0.5, density=True)

# Plot
plt.bar(k, pmf, width=0.8, alpha=0.6, label='Theoretical PMF')
plt.bar(k, counts, width=0.4, alpha=0.6, label='Empirical freq.')
plt.xlabel('k')
plt.ylabel('Probability')
plt.title(f'Geometric(p={p}) – theoretical vs. sampled')
plt.legend()
plt.show()

Negative Binomial


In [ ]:
from scipy.stats import nbinom
r, p, n_samples = 8, 0.3, 10_000
samples = negative_binomial(p, r, n_samples)
k = np.arange(samples.max() + 1)
empirical = np.bincount(samples, minlength=len(k)) / n_samples
plt.figure(figsize=(8, 4))
plt.bar(k, empirical, alpha=0.6, label="Sample frequencies")
plt.plot(k, nbinom.pmf(k, r, p), "o-", label="Theoretical PMF")
plt.xlabel("Failures before the eighth success")
plt.ylabel("Probability")
plt.title(f"Negative binomial (r={r}, p={p})")
plt.legend()
plt.tight_layout()
plt.show()

Hypergeometric


In [ ]:
N, K       = 50, 15     # population size N, with K successes (1's)
n_draw     = 10         # number drawn each sample
n_samples  = 10000      # how many hypergeometric samples
# build your “pre-sampled” pool
population = np.array([1]*K + [0]*(N-K))

# ─── SIMULATION ───────────────────────────────────────────────
# For each of n_samples: draw n_draw indices (no replacement), sum successes
hg_samples = [
    population[np.random.choice(N, n_draw, replace=False)].sum()
    for _ in range(n_samples)
]

# ─── THEORETICAL PMF ──────────────────────────────────────────
k = np.arange(0, min(K, n_draw) + 1)
theoretical_pmf = comb(K, k) * comb(N-K, n_draw - k) / comb(N, n_draw)

# ─── EMPIRICAL FREQUENCIES ───────────────────────────────────
empirical_freq = np.bincount(hg_samples, minlength=len(k)) / n_samples

# ─── PLOT ────────────────────────────────────────────────────
width = 0.35
plt.figure(figsize=(7,5))
plt.bar(k - width/2, theoretical_pmf, width, label='Theoretical PMF')
plt.bar(k + width/2, empirical_freq, width, label='Empirical PMF')
plt.xticks(k)
plt.xlabel('Number of Successes in the Draw (k)')
plt.ylabel('Probability')
plt.title(f'Hypergeometric(N={N}, K={K}, n={n_draw})')
plt.legend()
plt.tight_layout()
plt.show()

Part 2: Continuos

Uniform


In [ ]:
# 1) Parameters and sampling function
a, b = 0.0, 1.0
size = 100_000


# Draw samples
samples = uniform(a, b, size)

# 2) Plot empirical vs. theoretical
#   – theoretical PDF is constant = 1/(b-a) on [a,b]
x = np.linspace(a, b, 200)
pdf = np.ones_like(x) / (b - a)

plt.figure(figsize=(8, 5))
plt.hist(samples, bins=100, density=True, alpha=0.6, label='Empirical histogram')
plt.plot(x, pdf, lw=2, label='Theoretical PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.title(f'Uniform({a}, {b}): Theory vs. Samples')
plt.legend()
plt.show()

exponential


In [ ]:
size = 100_000
labda =1
a= 0
b =1

sample = exponential(labda,size)

x = np.linspace(0, sample.max(), 500)

theoretical_pdf = labda * np.exp(-labda * x)

plt.figure(figsize=(8, 5))
plt.hist(
    sample,
    bins=100,                  # moderate number of bins
    density=True,
    alpha=0.6,
    edgecolor='black',        # black borders
    linewidth=1.2,            # thicker edges
    label='Empirical histogram'
)
plt.plot(x, theoretical_pdf, lw=2, label='Theoretical PDF')
plt.xlabel('x')
plt.xlim(0,5)
plt.ylabel('Density')
plt.title(f'Exponential(λ={labda}): Theory vs. Samples')
plt.legend()
plt.show()

Gamma


In [ ]:
from math import gamma as gamma

size = 100_000
labda =1
a= 0
b =1

sample_1 = exponential(labda,size)
sample_2 = exponential(labda,size)
sample_3 = exponential(labda,size)
gamma_3 = sample_1+sample_2+sample_3



r = 3
x   = np.linspace(0, gamma_3.max(), 500)
pdf = (labda**r) * x**(r - 1) * np.exp(-labda * x) / gamma(r)

plt.figure(figsize=(8, 5))
plt.hist(
    gamma_3,
    bins=100,                  # moderate number of bins
    density=True,
    alpha=0.6,
    edgecolor='black',        # black borders
    linewidth=1.2,            # thicker edges
    label='Empirical histogram'
)

plt.plot(x, pdf, lw=2, label='Theoretical PDF')
plt.xlabel('x')
plt.xlim(0,10)
plt.ylabel('Density')
plt.title(f'Gamma (r={r}, labda = {labda}): Theory vs. Samples')
plt.legend()
plt.show()

Accept reject to get for other r values the Gamma distribution


In [ ]:
from scipy.stats import gamma as gamma_dist
shape, rate, num_sim = 3.4, 1.0, 10_000
samples, diagnostics = gamma_rejection(shape, rate, num_sim)
print(f"Envelope: {diagnostics['envelope']:.3f}")
print(f"Accepted {num_sim} of {diagnostics['proposals']} proposals")
x = np.linspace(0, 15, 400)
plt.figure(figsize=(8, 4))
plt.hist(samples, bins=50, density=True, alpha=0.6, label="Generated samples")
plt.plot(x, gamma_dist.pdf(x, a=shape, scale=1/rate), label="Theoretical PDF")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Gamma rejection sampling (shape=3.4, rate=1)")
plt.legend()
plt.tight_layout()
plt.show()

Beta


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

def beta_distribution(alpha_param, beta_param, lam=1.0, size=1):
    """
    Draw `size` samples from Beta(alpha_param, beta_param).

    Uses the fact that if
        X ~ Gamma(alpha_param,  1/lam) and
        Y ~ Gamma(beta_param,   1/lam)
    (independent), then  Z = X / (X + Y) ~ Beta(alpha_param, beta_param).
    """
    x = np.random.gamma(shape=alpha_param, scale=1 / lam, size=size)
    y = np.random.gamma(shape=beta_param,  scale=1 / lam, size=size)
    return x / (x + y)

# Example
alpha_ = 3
beta_  = 6
n      = 10_000

samples = beta_distribution(alpha_, beta_, size=n)

# quick visual check
x_grid = np.linspace(0, 1, 400)
plt.plot(x_grid, beta_dist.pdf(x_grid, alpha_, beta_), "k-", lw=2, label="True Beta PDF")
plt.hist(samples, bins=40, range=(0, 1), density=True, alpha=0.6, color="skyblue",
         label="Generated samples")
plt.title(f"Beta({alpha_}, {beta_}) via Gamma ratio")
plt.xlabel("x"); plt.ylabel("Density")
plt.legend(); plt.show()

Normal distribution


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Parameters
N = 10000

# 1) Generate the samples
samples = generate_normal_box_muller(N)

# 2) Plot histogram of samples vs. true N(0,1) PDF
plt.figure(figsize=(8, 5))
plt.hist(samples, bins=50, density=True, alpha=0.7, edgecolor='black')

# True PDF
x = np.linspace(-4, 4, 1000)
pdf = (1 / np.sqrt(2 * np.pi)) * np.exp(-x**2 / 2)
plt.plot(x, pdf, linewidth=2)

# Labels and title
plt.title('Box–Muller Generated Samples vs N(0,1) PDF')
plt.xlabel('x')
plt.ylabel('Density')
plt.grid(alpha=0.3)
plt.show()